# Continue Domain-Guided YOLO26L Training

**Purpose**: Resume training from epoch 112

**Changes from original training**:
1. Resume from `last.pt` checkpoint
2. Enable stronger augmentation (mosaic, mixup, etc.)
3. Adjust learning rate for fine-tuning phase

**Dataset**: https://www.kaggle.com/datasets/mfazrinizar/domain-guided-fetal-head-seg

In [ ]:
# Cell 1: Environment Setup
import os
import sys
import shutil
from pathlib import Path

# Detect environment
IS_KAGGLE = os.path.exists('/kaggle/input')
IS_COLAB = 'google.colab' in sys.modules

print(f"Environment: {'Kaggle' if IS_KAGGLE else 'Colab' if IS_COLAB else 'Local'}")

if IS_KAGGLE:
    # Install YOLO26 (custom ultralytics fork)
    print("Installing YOLO26...")
    !pip install -q git+https://github.com/mfazrinizar/ultralytics.git
    !pip install -q albumentations
    
    # Clear any memory leaks
    import gc
    import torch
    gc.collect()
    torch.cuda.empty_cache()
else:
    print("Using local environment")

In [ ]:
# Cell 2: Copy Dataset from Input to Working Directory
# Dataset: https://www.kaggle.com/datasets/mfazrinizar/domain-guided-continued
# Contains:
#   - fetal-head-segmentation/ (repo with augmented data)
#   - results/domain_guided_offline_l_continued/ (v2 checkpoint)

if IS_KAGGLE:
    import kagglehub
    path = kagglehub.dataset_download("mfazrinizar/domain-guided-continued")
    print("Path to dataset files:", path)

    INPUT_DATASET = Path('/kaggle/input/datasets/mfazrinizar/domain-guided-continued')
    WORKING_DIR = Path('/kaggle/working')
    
    print("Available in input dataset:")
    !ls -la {INPUT_DATASET}/
    
    # Clone fresh repo from GitHub (to get latest code fixes)
    REPO_DIR = WORKING_DIR / 'fetal-head-segmentation'
    if REPO_DIR.exists():
        print(f"\nRemoving stale repo: {REPO_DIR}")
        shutil.rmtree(REPO_DIR)
    
    print("\nCloning fresh from GitHub...")
    !git clone --depth 1 https://github.com/mfazrinizar/fetal-head-segmentation.git {REPO_DIR}
    
    # Copy data directory from dataset (not in git repo)
    DATA_SRC = INPUT_DATASET / 'fetal-head-segmentation' / 'data'
    DATA_DST = REPO_DIR / 'data'
    if DATA_SRC.exists() and not DATA_DST.exists():
        print("\nCopying data directory from dataset...")
        shutil.copytree(DATA_SRC, DATA_DST)
        print(f"Copied data to {DATA_DST}")
    
    # Install repo as package
    !cd {REPO_DIR} && pip install -q -e .
    
    # Copy results directory (contains checkpoint)
    RESULTS_DIR = WORKING_DIR / 'results'
    RESULTS_DIR.mkdir(exist_ok=True)
    
    # Copy the continued checkpoint
    CHECKPOINT_DIR = RESULTS_DIR / 'domain_guided_offline_l_continued'
    if not CHECKPOINT_DIR.exists():
        print("\nCopying checkpoint...")
        shutil.copytree(
            INPUT_DATASET / 'results' / 'domain_guided_offline_l_continued',
            CHECKPOINT_DIR
        )
        print(f"Copied to {CHECKPOINT_DIR}")
    else:
        print(f"\nCheckpoint already exists: {CHECKPOINT_DIR}")
    
    # Verify checkpoint
    CHECKPOINT_PATH = CHECKPOINT_DIR / 'weights' / 'last.pt'
    if CHECKPOINT_PATH.exists():
        print(f"\n[OK] Checkpoint: {CHECKPOINT_PATH} ({CHECKPOINT_PATH.stat().st_size/1e6:.1f}MB)")
    else:
        print(f"\n[ERROR] Checkpoint not found: {CHECKPOINT_PATH}")
        !ls -la {CHECKPOINT_DIR}/weights/ 2>/dev/null || echo "No weights directory"
else:
    REPO_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
    RESULTS_DIR = REPO_DIR / 'results'
    CHECKPOINT_DIR = RESULTS_DIR / 'domain_guided_offline_l_continued'
    CHECKPOINT_PATH = CHECKPOINT_DIR / 'weights' / 'last.pt'

In [ ]:
# Cell 3: Setup Python Path and Imports
import sys
sys.path.insert(0, str(REPO_DIR / 'src'))
sys.path.insert(0, str(REPO_DIR))

import yaml
import torch
import numpy as np
import pandas as pd
from ultralytics import YOLO

from src.util.constants import FETSAM_CLASS_WEIGHTS
from src.model.fetsam_loss_integration import (
    FetSAMBCEDiceLovaszLoss, patch_yolo_loss, unpatch_yolo_loss
)

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU count: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
print(f"FetSAM class weights: {FETSAM_CLASS_WEIGHTS}")

In [ ]:
# Cell 4: Analyze Previous Training Progress
results_csv = CHECKPOINT_DIR / 'results.csv'

if results_csv.exists():
    df = pd.read_csv(results_csv)
    df.columns = df.columns.str.strip()
    
    print("=== PREVIOUS TRAINING SUMMARY ===")
    print(f"Epochs completed: {len(df)}")
    print(f"Best mAP50 (Seg): {df['metrics/mAP50(M)'].max():.4f}")
    print(f"Best mAP50-95 (Seg): {df['metrics/mAP50-95(M)'].max():.4f}")
    print(f"\nLast 5 epochs mAP50-95: {df['metrics/mAP50-95(M)'].tail(5).values}")
    print(f"Mean: {df['metrics/mAP50-95(M)'].tail(5).mean():.4f}")
    
    LAST_EPOCH = len(df)
else:
    print("[WARNING] No results.csv found, assuming epoch 112")
    LAST_EPOCH = 112

In [ ]:
# Cell 5: Setup Data Directory and dataset.yaml
DATA_DIR = REPO_DIR / 'data'

# Verify data exists
train_images = DATA_DIR / 'train' / 'images'
if train_images.exists():
    n_train = len(list(train_images.glob('*.png')))
    print(f"[OK] Training images: {n_train}")
else:
    print(f"[ERROR] Training data not found: {train_images}")

# Create/update dataset.yaml
dataset_yaml = DATA_DIR / 'dataset.yaml'
dataset_config = {
    'path': str(DATA_DIR.absolute()),
    'train': 'train/images',
    'val': 'val/images',
    'test': 'test/images',
    'names': {0: 'Brain', 1: 'CSP', 2: 'LV'},
    'nc': 3
}

with open(dataset_yaml, 'w') as f:
    yaml.dump(dataset_config, f)
print(f"[OK] Dataset YAML: {dataset_yaml}")

In [ ]:
# Cell 6: Training Configuration
# Phase 3: Continue from v2 checkpoint (116 epochs done)
#
# ANALYSIS: Mosaic+mixup did NOT help break the 59% plateau.
# The model hit the same ceiling with or without mosaic/mixup.
#
# STRATEGY: Keep same settings, let cosine LR decay do the fine-tuning.
# LR is still ~0.01 - hasn't decayed yet. As it drops, model should
# converge more precisely. Expected: 60-62% mAP50-95.

class Config:
    RESUME_FROM = str(CHECKPOINT_PATH)
    REMAINING_EPOCHS = 72  # 188 - 116 already done
    
    # Hardware
    BATCH_SIZE = 16
    IMG_SIZE = 800
    DEVICE = '0,1' if torch.cuda.device_count() > 1 else '0'
    WORKERS = 8
    
    # Output
    PROJECT = str(RESULTS_DIR)
    NAME = 'domain_guided_offline_l_v3'

cfg = Config()

# Keep same augmentation as v2 for consistency
ENHANCED_AUGMENTATION = {
    # Color/intensity (safe for ultrasound)
    'hsv_h': 0.0,
    'hsv_s': 0.0,
    'hsv_v': 0.4,
    
    # Geometric
    'degrees': 45.0,
    'translate': 0.2,
    'scale': 0.5,
    'shear': 0.0,
    'perspective': 0.0,
    'flipud': 0.5,      # INCREASED from 0.3
    'fliplr': 0.5,      # INCREASED from 0.3
    
    # Multi-image augmentation (NEW - key for escaping plateau)
    'mosaic': 0.5,      # ENABLED - combines 4 images
    'mixup': 0.2,       # ENABLED - blends 2 images
    'copy_paste': 0.0,  # Keep disabled (destroys anatomy)
    
    # Additional
    'erasing': 0.4,     # Random erasing for regularization
}

print("=== ENHANCED AUGMENTATION CONFIG ===")
for k, v in ENHANCED_AUGMENTATION.items():
    print(f"  {k}: {v}")

print(f"\n=== TRAINING CONFIG ===")
print(f"Resume from: {cfg.RESUME_FROM}")
print(f"Target epochs: {cfg.TOTAL_EPOCHS}")
print(f"Batch size: {cfg.BATCH_SIZE}")
print(f"Image size: {cfg.IMG_SIZE}")
print(f"Device: {cfg.DEVICE}")

In [ ]:
# Cell 7: Load Model from Checkpoint
print("Loading model from checkpoint...")
model = YOLO(cfg.RESUME_FROM)
print(f"[OK] Model loaded from {cfg.RESUME_FROM}")
print(f"Model type: {type(model)}")

In [ ]:
# Cell 7b: Apply FetSAM Loss (class-weighted BCE+Dice+Lovasz)
# Note: patch_yolo_loss() must be called BEFORE loading the model
# Since we're loading from checkpoint, the loss is already baked in.
# We patch it anyway to ensure it's applied for this session.
print("Patching YOLO with FetSAM loss...")
patch_yolo_loss()
print(f"[OK] FetSAM loss patched (weights: {FETSAM_CLASS_WEIGHTS})")

In [ ]:
# Cell 8: Resume Training with Enhanced Augmentation
print("Starting resumed training with enhanced augmentation...")
print("="*60)

results = model.train(
    # Data
    data=str(dataset_yaml),
    
    # NO resume=True - we want to apply NEW augmentation settings
    # Model weights are already loaded from last.pt checkpoint
    # This starts fresh training with enhanced augmentation
    epochs=cfg.REMAINING_EPOCHS,  # 188 more epochs
    
    # Hardware
    batch=cfg.BATCH_SIZE,
    imgsz=cfg.IMG_SIZE,
    device=cfg.DEVICE,
    workers=cfg.WORKERS,
    
    # Output
    project=cfg.PROJECT,
    name=cfg.NAME,
    exist_ok=True,
    
    # Optimizer - fresh schedule for fine-tuning
    optimizer='auto',
    lr0=0.001,
    lrf=0.01,
    cos_lr=True,
    warmup_epochs=3,
    
    # Training settings
    patience=50,
    save=True,
    save_period=10,
    plots=True,
    verbose=True,
    seed=42,
    
    # ENHANCED AUGMENTATION
    **ENHANCED_AUGMENTATION,
    
    # Close mosaic near end
    close_mosaic=20,
)

print("\n" + "="*60)
print("[OK] Training complete!")

In [ ]:
# Cell 9: Setup Comprehensive Evaluation
from src.postprocess.comprehensive_evaluation import (
    ComprehensiveEvaluator, run_inference_evaluation,
    evaluate_from_ultralytics_results, generate_per_class_report
)
import gc

# Find best weights
v3_dir = RESULTS_DIR / cfg.NAME
best_weights = v3_dir / 'weights' / 'best.pt'
if not best_weights.exists():
    best_weights = CHECKPOINT_DIR / 'weights' / 'best.pt'
    print(f"Using checkpoint best: {best_weights}")

assert best_weights.exists(), f"No best weights found: {best_weights}"
print(f"Best weights: {best_weights} ({best_weights.stat().st_size/1e6:.1f}MB)")

EVAL_OUTPUT = RESULTS_DIR / cfg.NAME / 'evaluation'
EVAL_OUTPUT.mkdir(parents=True, exist_ok=True)
print(f"Evaluation output: {EVAL_OUTPUT}")

In [ ]:
# Cell 9a: Analyze Training Results (from results.csv)
# This generates training curves, convergence plots, and summary
v3_results_csv = v3_dir / 'results.csv'
csv_to_analyze = v3_results_csv if v3_results_csv.exists() else CHECKPOINT_DIR / 'results.csv'

training_summary = evaluate_from_ultralytics_results(
    results_dir=csv_to_analyze.parent,
    data_yaml=dataset_yaml,
    output_dir=EVAL_OUTPUT / 'training_analysis'
)

In [ ]:
# Cell 9b: Comprehensive Inference Evaluation - TEST set (No TTA)
# Computes: IoU, Dice, mIoU, mDice, per-class P/R/F1/Specificity,
#           confusion matrix, mAP50, mAP50-95
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("Running inference evaluation on TEST set (no TTA)...")
test_results = run_inference_evaluation(
    model_path=best_weights,
    data_yaml=dataset_yaml,
    split='test',
    output_dir=EVAL_OUTPUT / 'test_no_tta',
    conf=0.25,
    iou=0.5
)

In [ ]:
# Cell 9c: Comprehensive Inference Evaluation - TEST set (With TTA)
# Custom TTA: identity + hflip + vflip, merged via NMS + weighted mask avg
from src.postprocess.tta import SegmentationTTA

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("Running inference evaluation on TEST set (with TTA)...")
print("Transforms: identity + horizontal flip + vertical flip")

tta_evaluator = ComprehensiveEvaluator(
    model_path=best_weights,
    data_yaml=dataset_yaml,
    img_size=cfg.IMG_SIZE,
    conf_threshold=0.25,
    iou_threshold=0.5
)

# Replace model with TTA wrapper (same .predict() interface)
tta_evaluator.model = SegmentationTTA(tta_evaluator.model)

(EVAL_OUTPUT / 'test_tta').mkdir(parents=True, exist_ok=True)
test_tta_results = tta_evaluator.evaluate_split(
    split='test',
    output_dir=EVAL_OUTPUT / 'test_tta'
)
generate_per_class_report(test_tta_results, EVAL_OUTPUT / 'test_tta', 'test_tta')

In [ ]:
# Cell 9d: Comprehensive Inference Evaluation - VAL set (No TTA)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("Running inference evaluation on VAL set (no TTA)...")
val_results = run_inference_evaluation(
    model_path=best_weights,
    data_yaml=dataset_yaml,
    split='val',
    output_dir=EVAL_OUTPUT / 'val_no_tta',
    conf=0.25,
    iou=0.5
)

In [ ]:
# Cell 9e: Side-by-Side Comparison: TTA vs No TTA
print("\n" + "="*80)
print("COMPARISON: TTA vs No TTA (TEST SET)")
print("="*80)

def fmt(v):
    return f"{v:.4f}"

print(f"\n{'Metric':<25} {'No TTA':>10} {'TTA':>10} {'Delta':>10}")
print("-"*60)

comparisons = [
    ('Accuracy', test_results.accuracy, test_tta_results.accuracy),
    ('Precision', test_results.macro_precision, test_tta_results.macro_precision),
    ('Recall', test_results.macro_recall, test_tta_results.macro_recall),
    ('Specificity', test_results.macro_specificity, test_tta_results.macro_specificity),
    ('F1-Score', test_results.macro_f1, test_tta_results.macro_f1),
    ('Det mAP50', test_results.det_mAP50, test_tta_results.det_mAP50),
    ('Det mAP50-95', test_results.det_mAP50_95, test_tta_results.det_mAP50_95),
    ('Det mIoU', test_results.det_mIoU, test_tta_results.det_mIoU),
    ('Seg mAP50', test_results.seg_mAP50, test_tta_results.seg_mAP50),
    ('Seg mAP50-95', test_results.seg_mAP50_95, test_tta_results.seg_mAP50_95),
    ('Seg mIoU', test_results.seg_mIoU, test_tta_results.seg_mIoU),
    ('Seg mDice', test_results.seg_mDice, test_tta_results.seg_mDice),
]

for name, v1, v2 in comparisons:
    delta = v2 - v1
    print(f"{name:<25} {fmt(v1):>10} {fmt(v2):>10} {delta:>+10.4f}")

# Per-class comparison
print(f"\n{'='*80}")
print("PER-CLASS COMPARISON (TEST)")
print(f"{'='*80}")

for cls_name in ['Brain', 'CSP', 'LV']:
    m1 = test_results.per_class.get(cls_name)
    m2 = test_tta_results.per_class.get(cls_name)
    if m1 and m2:
        print(f"\n{cls_name}:")
        print(f"  {'Metric':<15} {'No TTA':>10} {'TTA':>10} {'Delta':>10}")
        for metric in ['precision', 'recall', 'f1_score', 'iou', 'dice']:
            v1 = getattr(m1, metric)
            v2 = getattr(m2, metric)
            print(f"  {metric:<15} {fmt(v1):>10} {fmt(v2):>10} {v2-v1:>+10.4f}")

In [ ]:
# Cell 10: Compare All Phases
v2_csv = CHECKPOINT_DIR / 'results.csv'
v3_csv = RESULTS_DIR / cfg.NAME / 'results.csv'

if v2_csv.exists():
    df_v2 = pd.read_csv(v2_csv)
    df_v2.columns = df_v2.columns.str.strip()
    print(f"Phase 2 (v2): {len(df_v2)} epochs, best mAP50-95: {df_v2['metrics/mAP50-95(M)'].max():.4f}")

if v3_csv.exists():
    df_v3 = pd.read_csv(v3_csv)
    df_v3.columns = df_v3.columns.str.strip()
    print(f"Phase 3 (v3): {len(df_v3)} epochs, best mAP50-95: {df_v3['metrics/mAP50-95(M)'].max():.4f}")
    improvement = df_v3['metrics/mAP50-95(M)'].max() - df_v2['metrics/mAP50-95(M)'].max()
    print(f"Improvement over v2: {improvement:+.4f}")